# G1 Docket -- Interactive Quickstart & Developer Walkthrough
Welcome to **G1 Docket**. This notebook demonstrates connecting to the local cluster, verifying cryptographic contracts, evaluating inline data attestation tags, wrapping LangChain and LlamaIndex pipelines, enforcing ABAC, auditing prompts pre-dispatch, and governing multi-cloud enterprise data stores (Snowflake, Databricks, Apache Iceberg).

## 1. Connect & Verify Cluster Health
We initialize `DocketClient` pointing to our local or remote cluster.

In [ ]:
from docket import DocketClient

client = DocketClient(base_url="http://localhost:18082")
health = client.get_health()
print(f"[+] Prevalidator Status: {health.get('status')} | Mode: {health.get('mode')} | Active Snapshot Sequence: {health.get('snapshot_sequence')}")

## 2. Inspect In-Memory Snapshot Contracts
Look up attestation records directly from the Prevalidator or Authoritative Registry.

In [ ]:
snapshot = client.get_latest_snapshot()
print(f"Snapshot Payload Type: {snapshot.get('payloadType')}")
print(f"Signatures present: {len(snapshot.get('signatures', []))}")

## 3. Inline Protection & Admission Gating
When prompts carry data wrapped in `<pod:object>` tags, the Prevalidator evaluates the caller key, content digest, and allowed egress routes in < 1 ms.

In [ ]:
from docket import PolicyDenialError

# A. Authorized Public Egress
public_prompt = (
    "Please summarize this verified document:\n"
    '<pod:object locator="file://estate/fileshare/product_overview_public.md">'
    "Open Source AI Architecture Overview"
    "</pod:object>"
)

resp = client.chat_completions(
    messages=[{"role": "user", "content": public_prompt}],
    model="gpt-4o",
    caller_key="pod-key-public",
)
print("Public Request Allowed:", resp["choices"][0]["message"]["content"])

# B. Unauthorized Sensitive Egress (Snowflake Restricted Table to Public LLM)
restricted_prompt = (
    "Analyze customer contacts:\n"
    '<pod:object locator="snowflake://contoso.us-east-1.snowflakecomputing.com/FINANCE_PROD/PUBLIC/DIM_CUSTOMER_CONTACTS">'
    "CUST-901 | elena.fisher@example.com | phone: 312-555-0199"
    "</pod:object>"
)

try:
    client.chat_completions(
        messages=[{"role": "user", "content": restricted_prompt}],
        model="gpt-4o",
        caller_key="pod-key-public",
    )
except PolicyDenialError as err:
    print(f"\n[!] Expected G1 Docket Refusal (HTTP {err.status_code}): {err.response_body.get('error', {}).get('message')}")

## 4. Protecting Agent Tools with `@attested_tool`
Decorate any tool function to automatically wrap its outputs in verified `<pod:object>` tokens with automatic SHA-256 calculation.

In [ ]:
from docket import attested_tool

@attested_tool("snowflake://analytics.corp.com/FINANCE/PUBLIC/Q3_REVENUE", auto_hash=True)
def get_sales_kpi(region: str) -> str:
    return f"{region} Revenue: \$18.4M | Growth: +24%"

tool_output = get_sales_kpi("EMEA")
print("Attested Tool Output:")
print(tool_output)

## 5. RAG Pipeline Transformers (LangChain & LlamaIndex)
Use drop-in pipeline components to automatically wrap retrieved documents or nodes.

In [ ]:
from docket import DocketDocumentTransformer, DocketLlamaIndexPostprocessor

# LangChain Document Transformer
class MockLangChainDoc:
    def __init__(self, content, source):
        self.page_content = content
        self.metadata = {"source": source}

transformer = DocketDocumentTransformer(auto_hash=True)
raw_docs = [MockLangChainDoc("Enterprise Architecture Specs", "s3://lake/specs.md")]
attested_docs = transformer.transform_documents(raw_docs)
print("LangChain Transformed Page Content:")
print(attested_docs[0].page_content)

## 6. Attribute-Based Access Control (ABAC) & Data Residency
Pass departmental claims and sovereign target regions to enforce zero-trust compartmentalization.

In [ ]:
# Cleared Finance Agent accessing Finance data
finance_prompt = (
    "Summarize verified financial context:\n"
    '<pod:object locator="file://estate/fileshare/product_overview_public.md">'
    "Open Source AI Architecture Overview"
    "</pod:object>"
)

approved_resp = client.chat_completions(
    messages=[{"role": "user", "content": finance_prompt}],
    model="gpt-4o",
    caller_key="pod-key-confidential",
    caller_dept="Finance",
    target_region="us-east-1",
)
print("Cleared Finance Prompt Response:", approved_resp["choices"][0]["message"]["content"])

## 7. Pre-Dispatch Prompt Auditing with LangChain Callback Handler
Enforce data attestation directly before prompt dispatch to an LLM, raising a 403 `PolicyDenialError` if unpapered data is used.

In [ ]:
from docket.integrations import DocketLangChainCallbackHandler
from docket import PolicyDenialError

handler = DocketLangChainCallbackHandler(require_attested_context=True, max_untagged_chars=100)

# 1. Allowed prompt with <pod:object> tag
valid_prompt = 'Analyze: <pod:object locator="s3://lake/specs.md">Attested Specs</pod:object>'
handler.on_llm_start(serialized={}, prompts=[valid_prompt])
print("[+] Valid Prompt Passed Callback Audit. Decision:", handler.audit_log[-1]["decision"])

# 2. Blocked prompt lacking required attestation tags
try:
    handler.on_llm_start(serialized={}, prompts=["Unpapered context without tags"])
except PolicyDenialError as err:
    print(f"[!] Prompt Blocked by Callback Handler (HTTP {err.status_code}):", err.response_body)

## 8. Multi-Cloud Warehouse Governance (Snowflake & Databricks Unity Catalog)
Gate and prevalidate queries against enterprise data warehouses with sub-millisecond in-memory verification and zero database calls on inference paths.

In [ ]:
# Query Snowflake and Databricks Unity Catalog via @attested_tool
from docket import DocketClient
from docket.integrations import attested_tool

SF_CATALOG = "snowflake://contoso.us-east-1.snowflakecomputing.com/FINANCE_PROD/PUBLIC/PUBLIC_PRODUCT_SKU_CATALOG"
DBX_STORE = "databricks://adb-41920194.azuredatabricks.net/main/sales_analytics/public_feature_store_registry"

@attested_tool(SF_CATALOG, auto_hash=True)
def get_snowflake_product() -> str:
    return "SKU-100 | Quantum Router | $1,299.00 | Available"

@attested_tool(DBX_STORE, auto_hash=True)
def get_databricks_features() -> str:
    return "feature_id:churn_prediction_v4 | freshness_hours:6 | team:mlops-platform | status:available"

with DocketClient(base_url="http://localhost:18082") as client:
    multi_query = f"Snowflake Catalog: {get_snowflake_product()}\nDatabricks Feature Store: {get_databricks_features()}"
    resp = client.chat_completions(
        messages=[{"role": "user", "content": f"Analyze enterprise data assets:\n{multi_query}"}],
        model="gpt-4o",
        caller_key="pod-key-public",
    )
    print("[+] Multi-Warehouse Query Allowed (200 OK):")
    print("    Response:", resp["choices"][0]["message"]["content"])


## 9. Open Lakehouse Governance (Apache Iceberg REST Catalog)
Query Apache Iceberg open table formats via `@attested_tool` with automatic DSSE cryptographic hash verification and zero-DB in-memory admission control.

In [ ]:
# Query Apache Iceberg REST Catalog table via @attested_tool
from docket import DocketClient
from docket.integrations import attested_tool

ICEBERG_CATALOG = "iceberg://polaris.us-east-1.lakehouse.net/finance_analytics/curated_product_hierarchy"

@attested_tool(ICEBERG_CATALOG, auto_hash=True)
def get_iceberg_product_hierarchy() -> str:
    return "category:compute_instances | sku_prefix:CP-8X | tier:General_Purpose | compliance:ISO27001"

with DocketClient(base_url="http://localhost:18082") as client:
    iceberg_data = get_iceberg_product_hierarchy()
    resp = client.chat_completions(
        messages=[{"role": "user", "content": f"Analyze compute infrastructure certification: {iceberg_data}"}],
        model="gpt-4o",
        caller_key="pod-key-public",
    )
    print("[+] Apache Iceberg Lakehouse Query Allowed (200 OK):")
    print("    Response:", resp["choices"][0]["message"]["content"])


## 10. Dynamic In-Flight Field-Level Masking & Redaction
When prompts contain sensitive or confidential fields (customer emails, direct phone numbers, personal identifiers) that are permitted for analytical modeling under masked policy, G1 Docket redacts sensitive spans dynamically in-flight while forwarding non-sensitive analytical fields untouched.

In [ ]:
# Dynamic In-Flight Field-Level Masking via X-POD-Permit-Masked
from docket import DocketClient
from docket.integrations import attested_tool

ICEBERG_REVENUE = "iceberg://polaris.us-east-1.lakehouse.net/finance_analytics/fact_gl_revenue_ledger"

@attested_tool(ICEBERG_REVENUE, auto_hash=True)
def get_confidential_revenue_ledger() -> str:
    return "fiscal_quarter:2026-Q3 | revenue_usd:$42,150,000 | operating_margin:28.4% | region:Global"

with DocketClient(base_url="http://localhost:18082") as client:
    ledger_data = get_confidential_revenue_ledger()
    # With permit_masked=True, Prevalidator dynamically redacts sensitive fields rather than denying
    resp = client.chat_completions(
        messages=[{"role": "user", "content": f"Analyze quarterly revenue trends: {ledger_data}"}],
        model="gpt-4o",
        caller_key="pod-key-public",
        permit_masked=True,
    )
    print("[+] Dynamic Masking Query Allowed (200 OK):")
    print("    Response:", resp["choices"][0]["message"]["content"])


## 11. Real-Time Prometheus Observability & Telemetry
Inspect live sub-millisecond evaluation latency, policy block rates, and active snapshot sequence numbers via the native `/metrics` endpoint.

In [ ]:
import httpx

metrics_resp = httpx.get("http://localhost:18082/metrics")
print(f"[+] Prometheus HTTP Status: {metrics_resp.status_code}")
print("[+] Sample Metrics Exposition:")
for line in metrics_resp.text.strip().splitlines()[:15]:
    print("   ", line)


## 12. Multi-Agent Cross-Clearance Orchestration

In modern multi-agent systems (LangGraph, AutoGen, CrewAI), specialized agents collaborate across security tiers:
* **DataWorker-01** (`pod-key-confidential`): Privileged database crawler querying internal Snowflake/BigQuery tables.
* **ComplianceAuditor-02** (`pod-key-analyst`): Reviews and anonymizes enterprise records.
* **PublicComms-03** (`pod-key-public`): Drafts customer-facing press releases and marketing content.

The G1 Docket Prevalidator mediates every agent handoff in < 0.2 ms, ensuring privileged data cannot leak to external cloud models without dynamic sanitization.

In [ ]:
from docket.integrations import attested_tool

SNOWFLAKE_BASE = 'snowflake://contoso.us-east-1.snowflakecomputing.com/FINANCE_PROD/PUBLIC'

# 1. Privileged Worker queries internal Snowflake CRM
@attested_tool(f'{SNOWFLAKE_BASE}/DIM_CUSTOMER_CONTACTS', auto_hash=True)
def query_crm(cust_id: str) -> str:
    return f'{cust_id} | elena.fisher@example.com | phone: 312-555-0199 | segment: Enterprise'

crm_record = query_crm('CUST-901')
print('Attested Tool Output:')
print(crm_record)

# 2. Privileged worker executes on local air-gapped model (ALLOW)
worker_res = client.chat_completions(
    messages=[{'role': 'user', 'content': f'Summarize customer: {crm_record}'}],
    model='local-model',
    caller_key='pod-key-confidential',
)
print('\n[+] Worker Local Model Action: ALLOW (HTTP 200 OK)')
print('   ', worker_res['choices'][0]['message']['content'])

# 3. Public agent attempts to send raw confidential context to gpt-4o (HARD BLOCK)
try:
    client.chat_completions(
        messages=[{'role': 'user', 'content': f'Publish newsletter: {crm_record}'}],
        model='gpt-4o',
        caller_key='pod-key-public',
    )
except PolicyDenialError as err:
    print('\n[+] Public Agent External Egress: HARD BLOCK (HTTP 403 Forbidden)')
    print('    Denial Reason:', err.response_body.get('error', {}).get('message'))

# 4. Safe Handoff: In-flight dynamic PII sanitization (permit_masked=True)
sanitized_res = client.chat_completions(
    messages=[{'role': 'user', 'content': f'Publish newsletter: {crm_record}'}],
    model='gpt-4o',
    caller_key='pod-key-public',
    permit_masked=True,
)
print('\n[+] Sanitized Handoff Action: DYNAMIC MASKING (HTTP 200 OK)')
print('   ', sanitized_res['choices'][0]['message']['content'])

## 13. Cryptographic Prompt Watermarking & Forensic Attribution

G1 Docket can invisibly embed cryptographic provenance metadata directly into prompt context or LLM outputs using zero-width Unicode steganography (`\u200B`, `\u200C`, `\u200D`).

If a sensitive prompt or chat log is leaked into the wild, forensic investigators can mathematically prove:
1. Active DSSE snapshot sequence number.
2. Originating caller key identity.
3. Precise UTC timestamp.
4. Non-repudiation HMAC-SHA256 signature.

In [ ]:
from docket import embed_invisible_watermark, extract_invisible_watermark, has_invisible_watermark

raw_text = 'Confidential executive briefing: projected Q4 gross margin expansion.'
print(f'Original Text (Length: {len(raw_text)}):')
print(f'  "{raw_text}"')

# 1. Embed invisible cryptographic watermark
watermarked = embed_invisible_watermark(
    text=raw_text,
    snapshot_sequence=3168,
    caller_key='pod-key-analyst',
)
print(f'\nWatermarked Text (Visible Characters: {len(watermarked.replace("\u200B", "").replace("\u200C", "").replace("\u200D", ""))}):')
print(f'  "{watermarked}"')
print(f'  Contains Hidden Stego Characters: {has_invisible_watermark(watermarked)}')

# 2. Forensic Extraction & HMAC Verification
payload = extract_invisible_watermark(watermarked)
print('\n[+] Forensic Watermark Extraction Results:')
print(f'  - Snapshot Sequence : #{payload.snapshot_sequence}')
print(f'  - Caller Identity   : {payload.caller_key}')
print(f'  - Timestamp (UTC)   : {payload.timestamp_iso}')
print(f'  - Signature Valid   : {payload.signature_valid} (HMAC: {payload.signature})')

# 3. Anti-Tampering Guarantee: Modified binary bits fail HMAC verification
from docket.watermark import encode_watermark_bytes, decode_watermark_bytes
tampered_bytes = bytearray(encode_watermark_bytes(3168, 'pod-key-analyst'))
tampered_bytes[3] ^= 0xFF  # Flip bit in payload
tampered_payload = decode_watermark_bytes(bytes(tampered_bytes))
print(f'\n[+] Tampered Payload Verification: signature_valid = {tampered_payload.signature_valid} (REJECTED)')

## 14. Dynamic ABAC Policy Hooks & Runtime Reconfiguration

G1 Docket enables security operators to dynamically enable, disable, and configure Attribute-Based Access Control (ABAC) policy rules at runtime without taking the gateway offline or dropping active inference requests.

- **Sovereign Data Residency (GDPR)**: Restricts EU/GDPR sovereign data to approved European sovereign enclaves (`eu-central-1`, `eu-west-1`, etc.).
- **Departmental Isolation**: Prevents cross-departmental data leaks by enforcing strict ownership scoping.


In [ ]:
# Inspect active runtime ABAC policy hooks
active_hooks = client.list_policy_hooks()
print('[+] Initial Policy Hooks:', active_hooks)

# Dynamically enable Data Residency (GDPR)
client.configure_policy_hook('data_residency', action='enable', allowed_regions=['eu-central-1', 'eu-west-1'])

# Dynamically enable Departmental Isolation
client.configure_policy_hook('departmental_isolation', action='enable')

updated_hooks = client.list_policy_hooks()
print('[+] Updated Policy Hooks (Enforcing):', updated_hooks)

# Clear runtime policy hooks for subsequent standard runs
client.configure_policy_hook('clear')
print('[+] Final Policy Hooks (Cleared):', client.list_policy_hooks())


## 15. CrewAI & LangGraph Multi-Agent Orchestration

The SDK provides first-class adapters for modern agentic frameworks, ensuring zero-trust cryptographic data attestation across multi-agent collaborations:

- `DocketCrewAITool`: Wraps any CrewAI tool so its outputs are automatically wrapped in canonical `<pod:object>` tokens with SHA-256 fingerprinting.
- `@docket_langgraph_node`: Decorates LangGraph graph node functions to maintain DSSE data provenance across state transitions.


In [ ]:
from docket.integrations import DocketCrewAITool, docket_langgraph_node

# 1. CrewAI Tool Wrapping
def raw_db_query(item_id: str) -> str:
    return 'SKU-100 | Quantum Router | $1,299.00 | Available'

crew_tool = DocketCrewAITool(raw_db_query, locator='snowflake://contoso/catalog', auto_hash=True)
tool_output = crew_tool('SKU-100')
print('[+] CrewAI Attested Tool Output:')
print('   ', tool_output)

# 2. LangGraph State Node Decorator
@docket_langgraph_node(locator='snowflake://contoso/crm', state_key='messages', auto_hash=True)
def retrieve_contact(state: dict) -> dict:
    messages = list(state.get('messages', []))
    messages.append({'role': 'user', 'content': 'CUST-901 | elena.fisher@example.com | phone: 312-555-0199 | segment: Enterprise'})
    return {'messages': messages}

state_out = retrieve_contact({'messages': []})
print('\n[+] LangGraph Node Attested Output:')
print('   ', state_out['messages'][0]['content'])
